In [7]:
using CSV, DataFrames, Dates, JSON, XLSX, StatsBase
function time_to_minutes(time)
    return Dates.hour(time) * 60 + Dates.minute(time)
end

time_to_minutes (generic function with 1 method)

In [15]:
file = "INSTANCES/OAMRP-Data/60-aircraft/AustralianTwoC/Flights.xlsx"
df_flights = DataFrame(XLSX.readtable(file, "Flights-60"))
df_flights.YEAR = year.(df_flights.Date)
df_flights.MONTH = month.(df_flights.Date)
df_flights.DAY = day.(df_flights.Date)
df_flights
select!(df_flights, Not(:Date))
rename!(df_flights, :Departure => :DEPARTURE_TIME)
rename!(df_flights, :Arrival => :ARRIVAL_TIME)
rename!(df_flights, :From => :ORIGIN_AIRPORT)
rename!(df_flights, :To => :DESTINATION_AIRPORT)

Row,Index,ORIGIN_AIRPORT,DEPARTURE_TIME,DESTINATION_AIRPORT,ARRIVAL_TIME,Day Change,YEAR,MONTH,DAY
,Any,Any,Any,Any,Any,Any,Int64,Int64,Int64
1,0,HIS,19:41:00,EMD,20:07:00,0,1999,12,31
2,1,EMD,19:49:00,WEI,21:18:00,0,1999,12,31
3,2,EMD,19:52:00,GKL,20:11:00,0,1999,12,31
4,3,EMD,20:00:00,MEL,21:43:00,0,1999,12,31
5,4,AZB,20:06:00,WEI,21:01:00,0,1999,12,31
6,5,MEL,20:08:00,EMD,21:51:00,0,1999,12,31
7,6,AZB,20:10:00,WEI,21:04:00,0,1999,12,31
8,7,MEL,20:12:00,EMD,21:55:00,0,1999,12,31
9,8,EMD,20:15:00,ADL,21:56:00,0,1999,12,31


In [16]:
for row in eachrow(df_flights)
    dt = time_to_minutes(row.DEPARTURE_TIME) + 1440*(row.DAY-1)
    at = time_to_minutes(row.ARRIVAL_TIME) + 1440*(row.DAY-1)
    if at < dt
        at += 1440
    end 
    row.DEPARTURE_TIME = dt
    row.ARRIVAL_TIME = at 
end 
df_flights.AIR_TIME = df_flights.ARRIVAL_TIME .- df_flights.DEPARTURE_TIME
df_flights
#XLSX.writetable(airline*"_2024-0"*string(m)*"_real_flights.xlsx", df_flights, overwrite = true)

Row,Index,ORIGIN_AIRPORT,DEPARTURE_TIME,DESTINATION_AIRPORT,ARRIVAL_TIME,Day Change,YEAR,MONTH,DAY,AIR_TIME
,Any,Any,Any,Any,Any,Any,Int64,Int64,Int64,Int64
1,0,HIS,44381,EMD,44407,0,1999,12,31,26
2,1,EMD,44389,WEI,44478,0,1999,12,31,89
3,2,EMD,44392,GKL,44411,0,1999,12,31,19
4,3,EMD,44400,MEL,44503,0,1999,12,31,103
5,4,AZB,44406,WEI,44461,0,1999,12,31,55
6,5,MEL,44408,EMD,44511,0,1999,12,31,103
7,6,AZB,44410,WEI,44464,0,1999,12,31,54
8,7,MEL,44412,EMD,44515,0,1999,12,31,103
9,8,EMD,44415,ADL,44516,0,1999,12,31,101


In [17]:
df_flights = df_flights[:, ["YEAR", "MONTH", "DAY", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DEPARTURE_TIME", "AIR_TIME", "ARRIVAL_TIME"]]

Row,YEAR,MONTH,DAY,ORIGIN_AIRPORT,DESTINATION_AIRPORT,DEPARTURE_TIME,AIR_TIME,ARRIVAL_TIME
,Int64,Int64,Int64,Any,Any,Any,Int64,Any
1,1999,12,31,HIS,EMD,44381,26,44407
2,1999,12,31,EMD,WEI,44389,89,44478
3,1999,12,31,EMD,GKL,44392,19,44411
4,1999,12,31,EMD,MEL,44400,103,44503
5,1999,12,31,AZB,WEI,44406,55,44461
6,1999,12,31,MEL,EMD,44408,103,44511
7,1999,12,31,AZB,WEI,44410,54,44464
8,1999,12,31,MEL,EMD,44412,103,44515
9,1999,12,31,EMD,ADL,44415,101,44516


In [19]:
XLSX.writetable("INSTANCES/OAMRP-Data/60-aircraft/AustralianTwoC/2078FL_60A.xlsx", df_flights, overwrite = true)
